In [1]:
ffmpeg_path = "/opt/homebrew/bin/ffmpeg"
ffprobe_path = "/opt/homebrew/bin/ffprobe"
FONT_PATH = "/System/Library/Fonts/Supplemental/Arial.ttf"

In [2]:
! $ffmpeg_path -version

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
built with Apple clang version 17.0.0 (clang-1700.0.13.3)
configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

In [3]:
import pandas as pd
import numpy as np
import glob 
import subprocess
import json
import os
import re
from collections import deque

In [ ]:

filenames = [
    # "batch_20250317_1538_pilotA.scienceData.jsonl",
    # "pilot/revision_202503/batch_20250618_1821_pilot.scienceData.jsonl",
    # "pilot/revision_202508/batch_20250903_1711_pilot.scienceData.jsonl"
    # "pilot/revision_202509/batch_20250915_2010_pilot.scienceData.jsonl"
    'pilot/revision_202509/batch_20251006_1952_pilot.scienceData.jsonl'
]

data = []
for filename in filenames:
    with open(filename) as f:
        for line in f:
            parsed = json.loads(line)
            if parsed['sampleId'] != 'missing':
                data.append(parsed)



collector = []
for row in data:
    gameId = row['gameId']
    position = row["position"]
    recordingIds = row['recordingIds']
    if recordingIds == "missing":
        continue

    for recordingId in recordingIds:
        collector.append({
            "gameId": gameId,
            "position": position,
            "recordingId": recordingId,
            "recordingsFolder": row["recordingsFolder"],
            "sampleId": row["sampleId"]
        })
        
recording_ids_df = pd.DataFrame(collector).set_index("recordingId")
recording_ids_df

    





,gameId,position,recordingsFolder,sampleId
recordingId,,,,
b65416eb-ec2d-475a-8108-55f37f367e18,01K6XJ2CG2E7KY9A8GA8S7FCRQ,1,20251006_1952_pilotS7FCRQ,48187bc9-80cc-4886-bc97-e79ff9325651
f991dbbc-ff66-419f-a1db-c76357bc558e,01K6XJ2CG2E7KY9A8GA8S7FCRQ,1,20251006_1952_pilotS7FCRQ,48187bc9-80cc-4886-bc97-e79ff9325651
762bf1bf-5aa4-48ee-bf54-9491aab53ba0,01K6XJ2CG2E7KY9A8GA8S7FCRQ,1,20251006_1952_pilotS7FCRQ,48187bc9-80cc-4886-bc97-e79ff9325651
c0537f7f-20da-4447-a7f7-4bab87962874,01K6XJ2CG2E7KY9A8GA8S7FCRQ,1,20251006_1952_pilotS7FCRQ,48187bc9-80cc-4886-bc97-e79ff9325651
755c27eb-f247-4018-b3f5-859090b4fb23,01K6XJ2CG2E7KY9A8GA8S7FCRQ,1,20251006_1952_pilotS7FCRQ,48187bc9-80cc-4886-bc97-e79ff9325651
...,...,...,...,...
239a528c-9156-48f3-b66c-15d702a399b9,01K6XPRBVN4Y4FAEAB8Q5VKRHH,1,20251006_1952_pilot5VKRHH,2e76230e-5f5e-4941-8411-af0032fee599
0007b6d0-5b2a-4099-9a98-22a3a0d165fa,01K6XPRBVN4Y4FAEAB8Q5VKRHH,1,20251006_1952_pilot5VKRHH,2e76230e-5f5e-4941-8411-af0032fee599
8dafa4df-4980-40db-bb2d-098131c21108,01K6XPRBVN4Y4FAEAB8Q5VKRHH,1,20251006_1952_pilot5VKRHH,2e76230e-5f5e-4941-8411-af0032fee599


In [5]:
recordings_folder = "recordings"
files = glob.glob(f"{recordings_folder}/**/*.webm")
print(f"Found {len(files)} files in {recordings_folder}")

collector = []
for file in files:
    if "normalized" in file: continue
    try:
        folder, filename = file.split("/")[-2:]
        date, time, batchGroup = folder.split("_")
        group = batchGroup.replace("disagr", "")
        mtgStartTime, *sessionId, _, filetype, trackStartTime = filename.split("-")
        sessionId = "_".join(sessionId)
        game, position, sampleId = -1,-1,None
        for recordingId in recording_ids_df.index:
            if recordingId in file:
                gameId = recording_ids_df.loc[recordingId]["gameId"]
                position = recording_ids_df.loc[recordingId]["position"]
                sampleId = recording_ids_df.loc[recordingId]["sampleId"]
                break
        
        collector.append({
            "path": file,
            "folder": folder,
            "filename": filename,
            "date": date,
            "time": time,
            "mtgStartTime": mtgStartTime,
            "sessionId": sessionId,
            "group": group,
            "filetype": filetype,
            "gameId": gameId,
            "position": int(position),
            "sampleId": sampleId,
        })
        
    except Exception as e:
        raise
        print(f"Error processing file {file}: {e}")
        continue
    


recordings_df = pd.DataFrame(collector).sort_values("path").dropna(subset="sampleId").reset_index(drop=True)

pd.options.display.max_colwidth = 150
recordings_df.to_csv("recordings/recordings_list.csv", index=False)
recordings_df


Found 642 files in recordings


,path,folder,filename,date,time,mtgStartTime,sessionId,group,filetype,gameId,position,sampleId
0,recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-audio-1759783042406.webm,20251006_1952_pilot3EZ1YD,1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-audio-1759783042406.webm,20251006,1952,1759783041402,70b35038_8e63_4dab_97c5_efbec2978411,pilot3EZ1YD,audio,01K6XKV9N97KVNDZ0ATW3EZ1YD,1,ab065315-4167-4d74-91bd-4c2eba1a2491
1,recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-video-1759783042734.webm,20251006_1952_pilot3EZ1YD,1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-video-1759783042734.webm,20251006,1952,1759783041402,70b35038_8e63_4dab_97c5_efbec2978411,pilot3EZ1YD,video,01K6XKV9N97KVNDZ0ATW3EZ1YD,1,ab065315-4167-4d74-91bd-4c2eba1a2491
2,recordings/20251006_1952_pilot3EZ1YD/1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-cam-audio-1759783042181.webm,20251006_1952_pilot3EZ1YD,1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-cam-audio-1759783042181.webm,20251006,1952,1759783041402,e8de2672_a87f_4bcb_a112_5d0cd98c5fe6,pilot3EZ1YD,audio,01K6XKV9N97KVNDZ0ATW3EZ1YD,0,eb127cec-2b0f-48b1-bf64-0079b0841f00
3,recordings/20251006_1952_pilot3EZ1YD/1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-cam-video-1759783042185.webm,20251006_1952_pilot3EZ1YD,1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-cam-video-1759783042185.webm,20251006,1952,1759783041402,e8de2672_a87f_4bcb_a112_5d0cd98c5fe6,pilot3EZ1YD,video,01K6XKV9N97KVNDZ0ATW3EZ1YD,0,eb127cec-2b0f-48b1-bf64-0079b0841f00
4,recordings/20251006_1952_pilot3EZ1YD/1759783141564-639a620e-fb6b-4d35-94e6-4e6dc2265c48-cam-audio-1759783142246.webm,20251006_1952_pilot3EZ1YD,1759783141564-639a620e-fb6b-4d35-94e6-4e6dc2265c48-cam-audio-1759783142246.webm,20251006,1952,1759783141564,639a620e_fb6b_4d35_94e6_4e6dc2265c48,pilot3EZ1YD,audio,01K6XKV9N97KVNDZ0ATW3EZ1YD,1,ab065315-4167-4d74-91bd-4c2eba1a2491
...,...,...,...,...,...,...,...,...,...,...,...,...
187,recordings/20251006_1952_pilotW9XPSA/1759784968922-524ff530-2539-49d0-9132-9a812adc0918-cam-video-1759784969415.webm,20251006_1952_pilotW9XPSA,1759784968922-524ff530-2539-49d0-9132-9a812adc0918-cam-video-1759784969415.webm,20251006,1952,1759784968922,524ff530_2539_49d0_9132_9a812adc0918,pilotW9XPSA,video,01K6XMYAQXJWZDTJG9CEW9XPSA,0,910b1b95-bbbb-42ee-807f-fffd12076616
188,recordings/20251006_1952_pilotW9XPSA/1759785214629-41392649-cda5-481b-a88d-49edae664b02-cam-audio-1759785216274.webm,20251006_1952_pilotW9XPSA,1759785214629-41392649-cda5-481b-a88d-49edae664b02-cam-audio-1759785216274.webm,20251006,1952,1759785214629,41392649_cda5_481b_a88d_49edae664b02,pilotW9XPSA,audio,01K6XMYAQXJWZDTJG9CEW9XPSA,1,5798d3e4-4c38-41a9-ac63-b50c8feea8d1
189,recordings/20251006_1952_pilotW9XPSA/1759785214629-41392649-cda5-481b-a88d-49edae664b02-cam-video-1759785216277.webm,20251006_1952_pilotW9XPSA,1759785214629-41392649-cda5-481b-a88d-49edae664b02-cam-video-1759785216277.webm,20251006,1952,1759785214629,41392649_cda5_481b_a88d_49edae664b02,pilotW9XPSA,video,01K6XMYAQXJWZDTJG9CEW9XPSA,1,5798d3e4-4c38-41a9-ac63-b50c8feea8d1
190,recordings/20251006_1952_pilotW9XPSA/1759785214629-96eade04-146a-4bb5-9c84-d3a758f1dd0d-cam-audio-1759785214919.webm,20251006_1952_pilotW9XPSA,1759785214629-96eade04-146a-4bb5-9c84-d3a758f1dd0d-cam-audio-1759785214919.webm,20251006,1952,1759785214629,96eade04_146a_4bb5_9c84_d3a758f1dd0d,pilotW9XPSA,audio,01K6XMYAQXJWZDTJG9CEW9XPSA,0,910b1b95-bbbb-42ee-807f-fffd12076616


In [6]:
"""
# Extracting Media Start Times

## Purpose:
This cell extracts the start time of each media file (video or audio) in the dataset. 
Since recordings may begin at different times due to participant delays or recording lags, 
we need to align them properly when merging. The start time will be used in later steps 
to correctly synchronize the media files.

## Process:
1. **`get_media_start_time(media_path, stream_type="v:0")`**:
   - Uses `ffprobe` to extract the start time of a given media file.
   - Supports both video (`v:0`) and audio (`a:0`) streams.
   - Returns the start time as a float (or `0` if unavailable).

2. **`apply_start_time(row)`**:
   - Determines whether the media file is audio or video.
   - Calls `get_media_start_time()` with the correct stream type.
   - Returns the extracted start time or `None` if an error occurs.

3. **Applying to the dataset**:
   - The `start_time` column is computed for each media file in `recordings_df`.
   - This ensures that each file's relative position in time is accounted for when 
     synchronizing and merging media files later in the pipeline.

## Why This Matters:
Without correcting for differing start times, the final merged video and audio 
could be misaligned, leading to desynchronized playback.
"""


def get_media_start_time(media_path, stream_type="v:0"):  
    """Extracts start time of a specific stream (video or audio)."""
    cmd = [
        ffprobe_path, "-v", "error", "-select_streams", stream_type,
        "-show_entries", "format=start_time",
        "-of", "json", media_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    metadata = json.loads(result.stdout)
    return float(metadata["format"]["start_time"]) if "format" in metadata else 0

def apply_start_time(row):
    try:
        media_path = row["path"]
        stream_type = "v:0" if row["filetype"] == "video" else "a:0"
        start_time = get_media_start_time(media_path, stream_type)
        return start_time
    except Exception as e:
        print(f"Error processing {row['path']}: {e}")
        return None
    


# Normalize Video Files

In [7]:


def check_normalized_video(video_path, expected_resolution="1280x720", expected_fps=24):
    """
    Performs integrity checks on a normalized video file to ensure it is valid and meets expected properties.
    
    **Checks performed:**
    1. **File existence** - Ensures the file is present at the expected location.
    2. **File size check** - Ensures the file is not empty (zero bytes).
    3. **File readability** - Attempts to decode the file to check if it is playable.
    4. **Duration validity** - Ensures the duration is greater than 0.
    5. **Resolution match** - Checks if the video resolution matches the expected resolution.
    6. **Frame rate check** - Verifies that the frame rate is within an acceptable margin of error.
    7. **Frame integrity checks (via `showinfo`)**:
       - **Sequential Frames Check**: Ensures `n` values are strictly increasing.
       - **PTS Order Check**: Ensures frame timestamps (`pts_time`) increase monotonically.
       - **Duplicate Frames Detection**: Identifies repeated frames by comparing `checksum`.
       - **Frozen Frame Detection**: Flags extended sequences of frames with identical `mean` values.
    8. **Overall Validation** - Returns a single `True`/`False` flag indicating whether the file is usable.

    **Returns:**
    A dictionary containing:
    - Results of each check (`True`/`False`).
    - A `"is_valid"` flag that summarizes whether the file passes all critical checks.
    - An `"errors"` field for debugging if the file is not valid.
    """

    results = {
        "exists": os.path.exists(video_path),
        "file_size": None,  # File size in bytes
        "readable": False,
        "duration_valid": False,
        "resolution_valid": False,
        "fps_valid": False,
        "frame_numbers_in_order": False,
        "pts_in_order": False,
        "duplicated_frames": False,
        "frozen_frames": False,
        
        "errors": [],
        "is_valid": False,  # Overall validation flag
    }

    # 1. **Check if the file exists**
    if not results["exists"]:
        results["errors"].append("File not found.")
        return results

    # 2. **Check if the file size is zero**
    file_size = os.path.getsize(video_path)
    results["file_size"] = file_size

    if file_size == 0:
        results["errors"].append("File exists but is empty (zero bytes).")
        return results  # Exit early

    # 3. **Check if the file is readable**
    cmd_readable = [
        ffmpeg_path, "-v", "error", "-i", video_path, "-f", "null", "-"
    ]
    readable_process = subprocess.run(cmd_readable, capture_output=True, text=True)

    if readable_process.returncode == 0:
        results["readable"] = True
    else:
        results["errors"].append("File exists but is unreadable.")
        return results  # Exit early

    # 4. **Extract video metadata (duration, resolution, fps)**
    cmd_metadata = [
        ffprobe_path, "-v", "error", "-select_streams", "v",
        "-show_entries", "stream=width,height,r_frame_rate,duration",
        "-of", "json", video_path
    ]
    result = subprocess.run(cmd_metadata, capture_output=True, text=True)

    if result.returncode != 0:
        results["errors"].append("FFprobe failed to read file.")
        return results

    try:
        metadata = json.loads(result.stdout)
        stream = metadata["streams"][0]

        # 5. **Check video duration**
        duration = float(stream.get("duration", 0))
        results["duration_valid"] = duration > 0  # Duration must be greater than zero

        # 6. **Check resolution**
        width, height = stream["width"], stream["height"]
        expected_width, expected_height = map(int, expected_resolution.split("x"))
        results["resolution_valid"] = (width, height) == (expected_width, expected_height)
        if not results["resolution_valid"]:
            results["errors"].append(f"Resolution mismatch: expected {expected_resolution}, got {width}x{height}")

        # 7. **Check frame rate**
        frame_rate = eval(stream["r_frame_rate"])  # Convert "30/1" to 30.0
        results["fps_valid"] = abs(frame_rate - expected_fps) < 0.5  # Allow small rounding error

    except Exception as e:
        results["errors"].append(f"Error parsing metadata: {e}")
        return results

    # 8. **Extract Frame Data using showinfo**
    cmd_showinfo = [
        ffmpeg_path, "-i", video_path, "-vf", "showinfo", "-f", "null", "-"
    ]
    result_showinfo = subprocess.run(cmd_showinfo, capture_output=True, text=True)

    if result_showinfo.returncode != 0:
        results["errors"].append("Failed to extract frame data.")
        return results

    frame_pattern = re.compile(
        r"\[Parsed_showinfo_\d+ @ 0x[0-9a-fA-F]+\]\s+n:\s*(\d+)\s+"
        r"pts:\s*(-?\d+)\s+pts_time:\s*([\d.]+)\s+"
        r"duration:\s*(-?\d+)\s+duration_time:\s*([\d.]+).*?"
        r"checksum:([0-9A-F]+).*?\s+"
        r"plane_checksum:\[[0-9A-F\s]+\]\s*"
        r"mean:\[\s*(\d+)\s+(\d+)\s+(\d+)\s*\]"
    )

    frame_numbers = []
    pts_times = []
    checksums = []
    mean_values = []

    for line in result_showinfo.stderr.split("\n"):
        match = frame_pattern.search(line)
        if match:
            frame_numbers.append(int(match.group(1)))
            pts_times.append(float(match.group(3)))
            checksums.append(match.group(6))
            mean_values.append([int(match.group(7)), int(match.group(8)), int(match.group(9))])

    # **Check for Out-of-Order Frames**
    results["frame_numbers_in_order"] = all(x < y for x, y in zip(frame_numbers, frame_numbers[1:]))

    # **Check for PTS Order**
    results["pts_in_order"] = all(x < y for x, y in zip(pts_times, pts_times[1:]))

    # **Check for Duplicated Frames (Identical Checksums)**
    results["duplicated_frames"] = any(x == y for x, y in zip(checksums, checksums[1:]))

    # **Check for Frozen Frames (Extended Sequences of Identical Mean Values)**
    frozen_frame_threshold = 20  # Flag runs of 20+ identical frames
    tolerance = 1  # Allow small variations

    recent_means = deque(maxlen=frozen_frame_threshold)
    current_sequence = []
    frozen_sequences = []

    for i, mean in enumerate(mean_values):
        if recent_means and all(abs(m - n) <= tolerance for m, n in zip(recent_means[-1], mean)):
            current_sequence.append(frame_numbers[i])
        else:
            if len(current_sequence) >= frozen_frame_threshold:
                frozen_sequences.append(current_sequence)
            current_sequence = [frame_numbers[i]]

        recent_means.append(mean)

    if len(current_sequence) >= frozen_frame_threshold:
        frozen_sequences.append(current_sequence)

    results["frozen_frames"] = len(frozen_sequences) > 0

    # 9. **Set Overall Validation Flag**
    results["is_valid"] = all([
        results["exists"], results["file_size"] > 0, results["readable"],
        results["duration_valid"], results["resolution_valid"], results["fps_valid"],
        results["frame_numbers_in_order"], results["pts_in_order"],
        # not results["duplicated_frames"], not results["frozen_frames"]
    ])

    return results

# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829.webm") # invalid, pre-normalized
# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-353873af-6c54-4510-bbcd-bd275feb611c-cam-video-1740788033829_normalized_start_66046.mp4")
# check_normalized_video("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-video-1740787968759_normalized_start_1193.mp4")

In [28]:



def normalize_video(video_path, start_time, position, game, sampleId, 
                    add_info=False, overwrite=False, 
                    FINAL_RESOLUTION = "1280:720",
                    FRAME_RATE = 24
                    ):
    """
    Normalizes video: Ensures fixed resolution and frame rate, adds optional text overlays.
    
    Displays "Video Muted" and/or "Audio Muted" based on mute events.
    """

    intermediate_path = video_path.replace(".webm", "_intermediate_fixed.mp4")
    output_path = video_path.replace(".webm", f"_normalized_start_{int(1000 * start_time)}.mp4")

    # **Step 1: Pre-check if the file already exists and is valid**
    if not overwrite:
        check_result = check_normalized_video(output_path)
        if check_result["is_valid"]:
            print(f"👉 Video already normalized: {output_path}")
            return output_path, None, check_result  # Skip processing

    # **Step 2: Convert WebM to MP4, ensuring resolution is fixed**
    cmd1 = [
        ffmpeg_path, "-y", "-hwaccel", "videotoolbox", "-threads", "0",
        "-i", video_path,
        "-vf", f"scale={FINAL_RESOLUTION}:flags=lanczos",
        "-c:v", "h264_videotoolbox", 
        #"-crf", str(FRAME_RATE), "-preset", "fast", # these options causing problems in macos26
        "-b:v", "2500k", "-q:v", "70",
        intermediate_path
    ]

    result1 = subprocess.run(cmd1, capture_output=True, text=True)
    if result1.returncode != 0:
        error_message = result1.stderr.strip()
        print(f"❌ FFmpeg Error (Step 1) for {video_path}:\n{error_message}")
        print("Command:", " ".join(cmd1))
        return None, error_message, None

    # **Step 3: Convert MP4 to final MP4, enforcing CFR and adding text overlays**
    video_filters = [f"fps={FRAME_RATE}"]

    # **Add Metadata Text Overlay (If Requested)**
    if add_info:
        metadata_text = f"Sample Id {sampleId}\nGroup {game}\nPosition {position}"
        video_filters.append(
            f"drawtext=fontfile='{FONT_PATH}':"
            f"text='{metadata_text}':x=10:y=10:fontcolor=white:fontsize=24:box=1:boxcolor=black@0.5"
        )


    # **Build FFmpeg Filter String**
    vf_filter = ",".join(video_filters)

    cmd2 = [
        ffmpeg_path, "-y", "-threads", "0", "-hwaccel", "videotoolbox",
        "-i", intermediate_path,
        "-vf", vf_filter,
        "-c:v", "h264_videotoolbox", 
        #"-crf", str(FRAME_RATE), "-preset", "fast", # these options causing problems in macos26
        "-b:v", "2500k", "-q:v", "70",
        output_path
    ]

    result2 = subprocess.run(cmd2, capture_output=True, text=True)
    if result2.returncode != 0:
        error_message = result2.stderr.strip()
        print(f"❌ FFmpeg Error (Step 2) for {video_path}:\n{error_message}")
        return None, error_message, None

    # **Step 4: Delete intermediate file**
    if os.path.exists(intermediate_path):
        os.remove(intermediate_path)
        print(f"🗑️ Deleted intermediate file: {intermediate_path}")

    # **Step 5: Validate Final Output**
    final_check = check_normalized_video(output_path)
    if not final_check["is_valid"]:
        print(f"⚠️ Warning: Final video may be corrupted at {output_path}", final_check)
        #return None, "Final output validation failed", final_check

    print(f"✅ Successfully processed {output_path}")
    return output_path, None, final_check  # No errors

# row = recordings_df.iloc[6]
# normalize_video(
#     video_path=row['path'], 
#     start_time=row['start_time'], 
#     position=row['position'],
#     game=row['game'],
#     sampleId=row['sampleId'],
#     muteEvents=row['muteEvents'],
#     add_info=True, 
#     overwrite=True
# )



In [9]:
import subprocess
import json
import os

def check_normalized_audio(audio_path):
    """
    Performs integrity checks on an audio file to ensure it is valid and meets expected properties.

    **Checks performed:**
    1. **File existence** - Ensures the file is present at the expected location.
    2. **File size check** - Ensures the file is not empty (zero bytes).
    3. **File readability** - Attempts to decode the file to check if it is playable.
    4. **Duration validity** - Ensures the duration is greater than 0.
    5. **Overall Validation** - Returns a single `True`/`False` flag indicating whether the file is usable.

    **Returns:**
    A dictionary containing:
    - Results of each check (`True`/`False`).
    - `"is_valid"` flag that summarizes whether the file passes all critical checks.
    - `"errors"` field for debugging if the file is not valid.
    """

    results = {
        "exists": os.path.exists(audio_path),
        "file_size": None,  # File size in bytes
        "readable": False,
        "duration_valid": False,
        "errors": None,
        "is_valid": False,  # Overall validation flag
    }

    # 1. **Check if the file exists**
    if not results["exists"]:
        results["errors"] = "File not found."
        return results

    # 2. **Check file size (ensure it's not empty)**
    file_size = os.path.getsize(audio_path)
    results["file_size"] = file_size

    if file_size == 0:
        results["errors"] = "File exists but is empty (zero bytes)."
        return results  # Exit early

    # 3. **Check if the file is readable**
    cmd_readable = [
        ffmpeg_path, "-v", "error", "-i", audio_path, "-f", "null", "-"
    ]
    readable_process = subprocess.run(cmd_readable, capture_output=True, text=True)

    if readable_process.returncode == 0:
        results["readable"] = True
    else:
        results["errors"] = "File exists but is unreadable."
        return results  # Exit early

    # 4. **Extract audio metadata (duration check)**
    cmd_metadata = [
        ffprobe_path, "-v", "error", "-show_entries", "format=duration",
        "-of", "json", audio_path
    ]
    result = subprocess.run(cmd_metadata, capture_output=True, text=True)

    if result.returncode != 0:
        results["errors"] = "FFprobe failed to read file."
        return results

    try:
        metadata = json.loads(result.stdout)
        duration = float(metadata["format"]["duration"]) if "format" in metadata and "duration" in metadata["format"] else 0
        results["duration_valid"] = duration > 0  # Duration must be greater than zero
    except Exception as e:
        results["errors"] = f"Error parsing metadata: {e}"
        return results

    # 5. **Set Overall Validation Flag**
    # The file is valid if all critical checks pass
    results["is_valid"] = (
        results["exists"]
        and results["file_size"] > 0
        and results["readable"]
        and results["duration_valid"]
    )

    return results

# Example usage
# check_normalized_audio("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764_normalized_start_240.mp3")

In [10]:
import os
import subprocess

def normalize_audio(audio_path, start_time, overwrite=False):
    """
    Normalize an audio file: Convert Opus to MP3, adjust timing so the audio starts at 0, and force mono.

    **Steps:**
    1. Check if a valid normalized file already exists.
    2. If overwrite is `False` and the file is valid, return existing file.
    3. Convert WebM audio to MP3, apply timing correction, and force mono.
    4. Validate the output to ensure it was correctly processed.
    
    **Returns:**
    - If successful: `(output_path, None)`
    - If failed: `(None, error_message)`
    """

    # Define output file path
    output_path = audio_path.replace(".webm", f"_normalized_start_{int(1000 * start_time)}.mp3")

    # **Check if the file is already valid**
    if not overwrite:
        audio_check = check_normalized_audio(output_path)
        if audio_check["is_valid"]:
            print(f"👉 Audio already normalized: {output_path}")
            return output_path, None, None  # Skip reprocessing

    # **FFmpeg command to normalize audio**
    cmd = [
        ffmpeg_path, "-y", 
        "-itsoffset", f"-{start_time}", "-i", audio_path,  # Shift start time to 0
        "-af", "pan=mono|c0=0.5*c0+0.5*c1",  # Convert stereo to mono
        "-c:a", "libmp3lame", "-b:a", "192k", output_path
    ]

    print(f"🔄 Processing audio: {audio_path}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    # **Check for errors**
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {audio_path}:\n{error_message}")
        return None, error_message, None # Return failure

    # **Validate the output file**
    final_check = check_normalized_audio(output_path)
    if not final_check["is_valid"]:
        print(f"❌ Audio validation failed after processing: {output_path}")
        return None, "Post-processing validation failed.", final_check

    print(f"✅ Audio successfully normalized: {output_path}")
    return output_path, None, final_check  # Success

# Example usage
# normalize_audio("recordings/20250301_0011_demo_QPK9NM/1740787968485-4395b77a-c5d1-482a-b413-ca212d06fc0c-cam-audio-1740787968764.webm", 0.24)

In [11]:
def get_media_duration(file_path):
    """Extracts duration of a media file (audio or video) using ffprobe."""

    if not isinstance(file_path, str) or not os.path.exists(file_path):  # Check if file path is valid
        print(f"Invalid file path: {file_path}")
        return None  # Skip invalid file paths

    cmd = [
        ffprobe_path, "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json", file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    try:
        metadata = json.loads(result.stdout)
        return float(metadata["format"]["duration"])  # Duration in seconds
    except (json.JSONDecodeError, KeyError, TypeError):
        return None
    
# recordings_df["duration"] = recordings_df.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
# recordings_df["end_time"] = recordings_df["start_time"] + recordings_df["duration"]
# recordings_df   

# test_recordings_df["duration"] = test_recordings_df.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
# test_recordings_df["end_time"] = test_recordings_df["start_time"] + test_recordings_df["duration"]
# test_recordings_df

In [12]:
import pandas as pd

def check_overlapping_files(group_df):
    """
    Checks that within each group:
    - No video files at the same position overlap.
    - No audio files at the same position overlap.
    """

    # Sort by position, file type, and start time
    group_df = group_df.sort_values(by=["position", "filetype", "start_time"]).reset_index(drop=True)

    # Track overlap warnings
    overlap_warnings = []

    # Iterate over unique positions
    for position in group_df["position"].unique():
        for filetype in ["video", "audio"]:
            # Filter by position and file type
            subset_df = group_df[(group_df["position"] == position) & (group_df["filetype"] == filetype)].copy()

            # Compute end times
            subset_df["end_time"] = subset_df["start_time"] + subset_df["duration"]

            # Check for overlaps
            for i in range(1, len(subset_df)):
                prev_row = subset_df.iloc[i - 1]
                curr_row = subset_df.iloc[i]

                if curr_row["start_time"] < prev_row["end_time"]:
                    overlap_warnings.append(
                        f"❌ Warning: Overlapping {filetype} files detected at position {position} in group {group_df.iloc[0]['group']}.\n"
                        f"Files: {prev_row['filename']} ({prev_row['start_time']} - {prev_row['end_time']}) and "
                        f"{curr_row['filename']} ({curr_row['start_time']} - {curr_row['end_time']})."
                    )

    # Print warnings if any exist
    if overlap_warnings:
        print("\n".join(overlap_warnings))

# Apply the function to each group
# recordings_df = recordings_df.groupby("group", group_keys=False).apply(check_overlapping_files).reset_index(drop=True)

# just first 3 groups
# num_groups = 3  # Change this to however many groups you want
# first_n_groups = recordings_df["group"].unique()[:num_groups]

# (recordings_df[recordings_df["group"].isin(first_n_groups)]
#     .groupby("group", group_keys=False)
#     .apply(check_overlapping_files)
# )

# # apply to test video
# test_recordings_df.groupby("group", group_keys=False).apply(check_overlapping_files)


In [13]:
def generate_black_video(black_video_path, final_resolution, frame_rate, duration):
    """Check if black background video exists; if not, generate it."""
    if os.path.exists(black_video_path):
        print(f"Black video already exists: {black_video_path}")
        return  # Skip generation

    print(f"Generating black video: {black_video_path}")
    cmd = [
        ffmpeg_path,
        "-f", "lavfi",
        "-i", f"color=c=black:s={final_resolution}:r={frame_rate}",
        "-t", str(duration),
        "-c:v", "libx264",
        "-preset", "fast",
        "-y", black_video_path  # Overwrite only if missing
    ]

    print(f"Running command: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


black_file = recordings_folder+"/black_10min.mp4"
generate_black_video(black_video_path=black_file, final_resolution="2560x720", frame_rate=24, duration=600)

Black video already exists: recordings/black_10min.mp4


In [14]:
import pandas as pd
import subprocess
import os

# Constants
BLACK_VIDEO_PATH = recordings_folder+"black_10min.mp4"  # Pre-generated black background
FINAL_RESOLUTION = "2560:720"  # Ensure it fits side-by-side layout
final_width, final_height = map(int, FINAL_RESOLUTION.split(":"))

def merge_videos_for_group(group_df, black_video_path=BLACK_VIDEO_PATH):
    """
    Merge participant videos onto a black background, aligning them based on offsets.
    - Ensures position 0 is always on the left.
    - Ensures position 1 is always on the right.
    - Makes each video disappear instead of holding the last frame.
    """

    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    output_path = os.path.join(recordings_folder, group_df.iloc[0]["folder"], f"{mtgStartTime}_merged_video.mp4")

    # **Filter to only video files** (Exclude audio files)
    video_files = group_df[group_df["filetype"] == "video"]

    if video_files.empty:
        print(f"Skipping session {group_id} - No video files found.")
        return None

    # Ensure we have exactly two positions: left (0) and right (1)
    positions = sorted(video_files["position"].dropna().unique())
    if positions != [0, 1]:
        print(f"Skipping session {group_id} - Expected exactly 2 positions (0 and 1), found {positions}.")
        return None

    # Assign fixed positions
    position_map = {
        0: "(W/2-w)/2:(H-h)/2",      # Left position
        1: "(W/2-w)/2+W/2:(H-h)/2"   # Right position
    }

    # FFmpeg input files
    inputs = ["-i", black_video_path]  # Start with black background
    overlays = []
    video_index = 1  # Video input index in FFmpeg (excluding black background)

    for _, row in video_files.iterrows():
        if pd.isna(row["normalized_path"]):  # Skip missing videos
            continue

        start_time = row["start_time"]
        duration = row["duration"]  # Assuming you have the duration of each video
        position = position_map[row["position"]]  # Get left/right position

        # Add video input
        inputs.extend(["-i", row["normalized_path"]])

        # Scale video and adjust timestamp for correct alignment
        overlays.append(f"[{video_index}:v] setpts=PTS-STARTPTS+{start_time}/TB, scale={final_width//2}:-1 [v{video_index}];")

        # Ensure video disappears after its duration
        enable_filter = f"enable='between(t,{start_time},{start_time + duration})'"

        if video_index == 1:
            overlays.append(f"[0:v][v{video_index}] overlay={position}:{enable_filter} [out{video_index}];")
        else:
            overlays.append(f"[out{video_index-1}][v{video_index}] overlay={position}:{enable_filter} [out{video_index}];")

        video_index += 1

    # Build FFmpeg filter complex
    final_output_tag = f"[out{video_index-1}]"
    drawtext_filter = (
        f"{final_output_tag} drawtext=fontfile='{FONT_PATH}':"
        f"text='%{{pts\\:hms}}':"
        f"x=10: y=h-th-10: fontcolor=white: fontsize=20: box=1: boxcolor=black@0.5"
        f" [final_output]"
    )


    # filter_complex = " ".join(overlays)
    filter_complex = " ".join(overlays) + drawtext_filter

    cmd = [
        ffmpeg_path,
        "-hwaccel", "videotoolbox",  # ✅ Use Mac GPU for decoding
        "-threads", "0",  # ✅ Allow FFmpeg to optimize CPU usage
        *inputs,
        "-filter_complex", filter_complex,
        # "-map", final_output_tag,  # Ensure only the final video stream is mapped
         "-map", "[final_output]",  # Ensure only the final video stream is mapped

        # "-c:v", "libx264", 
        "-c:v", "h264_videotoolbox",  # 🚀 Enable GPU acceleration
        "-b:v", "5M",  # Required for GPU encoding
        "-preset", "fast",
        "-y", output_path  # Overwrite existing file
    ]

    # Run FFmpeg
    print(f"Processing merged video for session {group_id}...")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {output_path}", error_message)
        return None

    # print(f" Successfully processed {output_path}")
    return output_path

# Apply function to a test group
# merge_videos_for_group(test_recordings_df, black_video_path="recordings/black_2min.mp4")  # Run the function on the first group only


# # just first 3 groups
# num_groups = 100  # Change this to however many groups you want
# first_n_groups = recordings_df["group"].unique()[:num_groups]

# (recordings_df[recordings_df["group"].isin(first_n_groups)]
#     .groupby("group", group_keys=False)
#     .apply(merge_videos_for_group)
# )

# n_groups = 250  # Adjust as needed

# for name, group in recordings_df.groupby("group"):
#     if group['normalized_path'].isnull().sum() > 0:
#         print(f"Skipping group {name} - missing normalized files.")
#         continue
#     try:
#         merge_videos_for_group(group)
#         print(f"✅ Processed group {index+1}/{n_groups}")
#     except Exception as e:
#         print(f"❌ Error processing row {index+1}: {e}")

# print("✅ Processing complete.")


In [15]:
def generate_silence_if_needed(silence_audio_path, duration):
    """Generate a 15-minute silent MP3 file if it doesn't exist."""
    if not os.path.exists(silence_audio_path):
        print("Generating 15-minute silence file...")
        cmd = [
            ffmpeg_path, "-f", "lavfi", "-t", str(duration), "-i", "anullsrc=r=48000:cl=stereo",
            "-c:a", "libmp3lame", "-b:a", "192k", silence_audio_path
        ]
        subprocess.run(cmd, check=True)
    else:
        print("Silence file already exists.")

silence_file = recordings_folder+"/silence_10min.mp3"
generate_silence_if_needed(silence_file, 600)
# generate_silence_if_needed(recordings_folder+"silence_2min.mp3", 120)

Silence file already exists.


In [16]:
# this mostly works?
import pandas as pd
import subprocess
import os
import json

# Constants
SILENCE_AUDIO_PATH = recordings_folder+"silence_10min.mp3"  # Pre-generated silent audio file
FINAL_AUDIO_BITRATE = "192k"

def is_mono(audio_path):
    """Check if an audio file is mono using ffprobe."""
    cmd = [
        ffprobe_path, "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=channels", "-of", "json", audio_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    try:
        metadata = json.loads(result.stdout)
        return metadata["streams"][0]["channels"] == 1  # True if mono
    except (json.JSONDecodeError, KeyError, IndexError):
        return False  # Assume not mono if any error occurs

def merge_audio_for_group(group_df, silence_audio_path=SILENCE_AUDIO_PATH, duration=900):
    """
    Merge participant audio files into a stereo track:
    - First participant -> Left channel (c0)
    - Second participant -> Right channel (c1)
    - Properly aligns based on start offsets
    """
    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    output_path = os.path.join(recordings_folder, group_df.iloc[0]["folder"], f"{mtgStartTime}_merged_audio.mp3")

    # **Filter to only audio files** (Exclude video files)
    audio_files = group_df[group_df["filetype"] == "audio"]

    if audio_files.empty:
        print(f"Skipping session {group_id} - No audio files found.")
        return None

    # Get unique participant IDs
    participants = sorted(audio_files["position"].dropna().unique())

    if len(participants) != 2:
        print(f"Skipping session {group_id} - Expected 2 participants, found {len(participants)}.")
        return None

    # Assign left (P1) and right (P2) channels
    participant_channels = {
        participants[0]: "c0",  # Left channel
        participants[1]: "c1"   # Right channel
    }

    # FFmpeg input files
    inputs = ["-i", silence_audio_path]  # Start with silent audio
    audio_filters = ["[0:a] acopy [a0];"]
    audio_index = 1  # Audio input index in FFmpeg

    for _, row in audio_files.iterrows():
        if pd.isna(row["normalized_path"]):  # Skip missing audio
            continue

        audio_path = row["normalized_path"]

        # **Check if audio is mono before proceeding**
        if not is_mono(audio_path):
            print(f"Error: {audio_path} is not mono. Please re-run normalization.")
            return None  # Stop processing if an issue is found

        start_time = row["start_time"]
        participant_id = row["position"]
        channel = participant_channels[participant_id]

        # Add audio input
        inputs.extend(["-i", audio_path])

        # Offset audio to align correctly and map to left or right
        end_pad = (duration-row["end_time"])*48000
        # audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad}, pan=stereo|{channel}=c0 [a{audio_index}];")
        # audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad}, pan=stereo|{channel}=0.5*c0+0.5*c1 [a{audio_index}];")
        audio_filters.append(f"[{audio_index}:a] adelay={int(start_time*1000)}, apad=pad_len={end_pad} [a{audio_index}];") # keep as mono
        audio_index += 1

    # Merge into a stereo track
    audio_tracks = [f"[a{i}]" for i in range(audio_index)]
    merged_audio = f"{''.join(audio_tracks)} amerge=inputs={len(audio_files)+1} [final_out];"

    # Build FFmpeg filter complex
    filter_complex = " ".join(audio_filters) + merged_audio
    final_output_tag = "[final_out]"

    cmd = [
        ffmpeg_path,
        *inputs,
        "-filter_complex", filter_complex,
        "-map", final_output_tag,  # Ensure only the final audio stream is mapped
        "-c:a", "libmp3lame", "-b:a", FINAL_AUDIO_BITRATE,
        "-y", output_path  # Overwrite existing file
    ]

    # Run FFmpeg
    print(f"Processing merged audio for session {group_id}...")
    # print(cmd)

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.returncode != 0:
        error_message = result.stderr.strip()
        print(f"❌ FFmpeg Error for {output_path}:\n{error_message}")
        return None

    return output_path

# merge_audio_for_group(test_recordings_df, silence_audio_path="recordings/silence_2min.mp3", duration=120)  # Run the function on the first group only

# for name, group in recordings_df.groupby("group"):
#     if group['normalized_path'].isnull().sum() > 0:
#         print(f"👉 Skipping group {name} - missing normalized files.")
#         continue
#     try:
#         merge_audio_for_group(group)
#         print(f"✅ Processed group {name}")
#     except Exception as e:
#         print(f"❌ Error processing row {index+1}: {e}")

# print("✅ Processing complete.")

In [17]:
import subprocess
import os
import shutil

# Define the destination folder for finalized files
READY_TO_CODE_DIR = recordings_folder+"/ready_to_code/mid_top"

# Ensure the directory exists
os.makedirs(READY_TO_CODE_DIR, exist_ok=True)

def merge_audio_video_for_group(group_df, overwrite=False):
    """
    Merge the group's merged video and audio files into a final MP4 file.
    - Skips if the final output already exists.
    - Saves a copy in 'recordings/ready_to_code/' with a detailed filename.
    """
    group_id = group_df.iloc[0]["group"]
    mtgStartTime = group_df.iloc[0]["mtgStartTime"]
    date = group_df.iloc[0]["date"]
    time = group_df.iloc[0]["time"]
    batch = group_df.iloc[0].get("batch", "unknown")  # Ensure batch exists
    game_number = group_df.iloc[0].get("game", "unknown")  # Ensure game number exists
    folder_path = os.path.join(recordings_folder, group_df.iloc[0]["folder"])

    # Define expected input/output file paths
    video_path = os.path.join(folder_path, f"{mtgStartTime}_merged_video.mp4")
    audio_path = os.path.join(folder_path, f"{mtgStartTime}_merged_audio.mp3")
    new_filename = f"{date}_{time}_group_{group_id}_start_{mtgStartTime}.mp4"
    output_path = os.path.join(folder_path, new_filename)
    

    # Check if the final output already exists
    if not overwrite and os.path.exists(output_path):
        print(f"✔ Skipping group {group_id} - Final output already exists: {output_path}")
    else:
        # Check if required files exist before proceeding
        if not os.path.exists(video_path):
            print(f"❌ Skipping session {group_id} - Merged video not found: {video_path}")
            return None

        if not os.path.exists(audio_path):
            print(f"❌ Skipping session {group_id} - Merged audio not found: {audio_path}")
            return None

        # FFmpeg command to merge audio and video
        cmd = [
            ffmpeg_path,
            "-i", video_path,  # Input video
            "-i", audio_path,  # Input audio
            "-map", "0:v",  # Ensure we take video from first input
            "-map", "1:a",  # Ensure we take audio from second input
            "-c:v", "copy",  # Don't re-encode video
            "-c:a", "aac", "-b:a", "192k",  # Convert audio to AAC (for MP4 compatibility)
            "-y", output_path  # Overwrite if exists
        ]

        # Run FFmpeg
        # print(f"Processing final merged file for session {group_id}...")
        # print(f"\n\nCMD:\n{' '.join(cmd)}\n\n")
        
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if result.returncode != 0:
            error_message = result.stderr.strip()
            print(f"❌ FFmpeg Error for {output_path}:\n{error_message}")
            return None

        # print(f"✔ Merged {group_id}: {output_path}")

    # Copy the finalized file
    destination_path = os.path.join(READY_TO_CODE_DIR, new_filename)
    shutil.copy(output_path, destination_path)
    print(f"📂 Copied to: {destination_path}")

    return output_path



In [29]:
import datetime
overwrite = True # whether to overwrite normalized files. 



for name, group in recordings_df.groupby(["group", "mtgStartTime"]):

    print(f"\n\nProcessing group {name}, {group.iloc[0]['group']}")
    start = datetime.datetime.now()
    # if group['discussion_happened'].all() == False:
    #     print(f"Skipping group {name} - discussion did not happen.")
    #     continue

    folder_path = os.path.join(recordings_folder, group.iloc[0]["folder"])
    composed_file_exists = len(glob.glob(os.path.join(folder_path, "*_group_*.mp4"))) > 0
    if composed_file_exists and not overwrite:
        print(f"⏩ Skipping group - composed file already exists.")
        continue

    try:
        print("Get start times")
        group["start_time"] = group.apply(apply_start_time, axis=1)

        print("Normalize Video and Audio")
        for index in group.index:
            row = group.loc[index]
            

            if row["filetype"] == "video":
                normalized_path, error_msg, checks = normalize_video(row["path"], float(row["start_time"]), row["position"], row["gameId"], row["sampleId"], add_info=True, overwrite=overwrite)
                group.at[index, "normalized_path"] = normalized_path
                group.at[index, "error_msg"] = error_msg
                group.at[index, "duplicated_frames"] = checks["duplicated_frames"]
                group.at[index, "frozen_frames"] = checks["frozen_frames"]
                group.at[index, "is_valid"] = checks["is_valid"]
                if error_msg:
                    print(f"❌ Error processing video {row['path']}: {error_msg}")
       
            elif row["filetype"] == "audio":
                normalized_path, error_msg, checks = normalize_audio(row["path"], float(row["start_time"]), overwrite=overwrite)
                group.at[index, "normalized_path"] = normalized_path
                group.at[index, "error_msg"] = error_msg
                recordings_df.at[index, "duplicated_frames"] = pd.NA
                recordings_df.at[index, "frozen_frames"] = pd.NA
                group.at[index, "is_valid"] = check_normalized_audio(normalized_path)["is_valid"]
                if error_msg:
                    print(f"❌ Error processing audio {row['path']}: {error_msg}")

        print("Get audio durations")
        group["duration"] = group.apply(lambda row: get_media_duration(row["normalized_path"]), axis=1)
        group["end_time"] = group["start_time"] + group["duration"]

        print("Check overlapping files")
        check_overlapping_files(group)

        print("merge videos")
        merge_videos_for_group(group, black_video_path=black_file)

        print("merge audio")
        merge_audio_for_group(group, silence_audio_path=silence_file, duration=900)

        print("merge audio video")
        merge_audio_video_for_group(group, overwrite=overwrite)

        end = datetime.datetime.now()
        duration = end-start
        total_seconds = duration.total_seconds()
        minutes = int(total_seconds // 60)
        seconds = int(total_seconds % 60)

        print(f"✅ Processed group {name} in {minutes}:{seconds}")
    except Exception as e:
        raise
        print(f"❌ Error processing group {name}: {e}")
        # continue






Processing group ('pilot3EZ1YD', '1759783041402'), pilot3EZ1YD
Get start times
Normalize Video and Audio
🔄 Processing audio: recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-audio-1759783042406.webm
✅ Audio successfully normalized: recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-audio-1759783042406_normalized_start_963.mp3
🗑️ Deleted intermediate file: recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-video-1759783042734_intermediate_fixed.mp4
✅ Successfully processed recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-video-1759783042734_normalized_start_1259.mp4
🔄 Processing audio: recordings/20251006_1952_pilot3EZ1YD/1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-cam-audio-1759783042181.webm
✅ Audio successfully normalized: recordings/20251006_1952_pilot3EZ1YD/1759783041402-e8de2672-a87f-4bcb-a112-5d0cd98c5fe6-

In [ ]:
# files_to_delete = glob.glob(recordings_folder+"/*/*normalized_start*.mp*")
# for file in files_to_delete:
#     try:
#         os.remove(file)
#         print(f"🗑️ Deleted: {file}")
#     except Exception as e:
#         print(f"❌ Error deleting {file}: {e}")


In [23]:
! ffprobe recordings/20251006_1952_pilot3EZ1YD/1759783041402-70b35038-8e63-4dab-97c5-efbec2978411-cam-video-1759783042734.webm

ffprobe version 7.1.1 Copyright (c) 2007-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspee